# Proyecto completo de ML: modelo de aprobación de tarjetas de crédito

Una entidad financiera quiere **automatizar** una primera revisión 
de las **solicitudes** de tarjetas de crédito. Para ello dispone de 
**datos históricos** de clientes y de la **decisión** que se tomó con cada solicitud

### Dataset

Dataset de Kaggle `Credit Card Details`. 

Hay dos ficheros:

`Credit_card.csv`: datos de solicitantes
`Credit_card_label`: variable objetivo (0 = aprobada y 1 = rechazada)

### Objetivo 

Construir, con scikit-learn, un modelo de Regresión Logística integrado en un Pipeline que prediga, a partir de los datos de un cliente, si su solicitud de tarjeta será rechazada.

# Carga de datos

Descargamos los dos datasets. Los guardamos en data/raw en formato .csv y los cargamos.

`df`: dataset con las variables de los datos de los solicitantes
`df_target`: dataset con la vble respuesta (solicitudes aceptadas o rechazadas)

In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("rohitudageri/credit-card-details")

print("Path to dataset files:", path)

Path to dataset files: /home/estevoariasgarcia/.cache/kagglehub/datasets/rohitudageri/credit-card-details/versions/1


In [3]:
import os
import shutil

# Crear carpeta
os.makedirs("data/raw", exist_ok=True)

# Copiar os ficheiros orixinais
for archivo in ["Credit_card.csv", "Credit_card_label.csv"]:
    origen = os.path.join(path, archivo)
    destino = os.path.join("data/raw", archivo)
    shutil.copy2(origen, destino)

In [4]:
import pandas as pd
import  os

df = pd.read_csv(os.path.join(path, "Credit_card.csv"))
df_label = pd.read_csv(os.path.join(path, "Credit_card_label.csv"))

display(df.head())
display(df_label.head())

,Ind_ID,GENDER,Car_Owner,Propert_Owner,CHILDREN,Annual_income,Type_Income,EDUCATION,Marital_status,Housing_type,Birthday_count,Employed_days,Mobile_phone,Work_Phone,Phone,EMAIL_ID,Type_Occupation,Family_Members
0,5008827,M,Y,Y,0,180000.0,Pensioner,Higher education,Married,House / apartment,-18772.0,365243,1,0,0,0,NaN,2
1,5009744,F,Y,N,0,315000.0,Commercial associate,Higher education,Married,House / apartment,-13557.0,-586,1,1,1,0,NaN,2
2,5009746,F,Y,N,0,315000.0,Commercial associate,Higher education,Married,House / apartment,NaN,-586,1,1,1,0,NaN,2
3,5009749,F,Y,N,0,NaN,Commercial associate,Higher education,Married,House / apartment,-13557.0,-586,1,1,1,0,NaN,2
4,5009752,F,Y,N,0,315000.0,Commercial associate,Higher education,Married,House / apartment,-13557.0,-586,1,1,1,0,NaN,2


,Ind_ID,label
0,5008827,1
1,5009744,1
2,5009746,1
3,5009749,1
4,5009752,1


## Merge para obtener un df único.

In [24]:
print("Columnas de df:")
print(df.columns.tolist())

print("\nColumnas de df_label:")
print(df_label.columns.tolist())

Columnas de df:
['Ind_ID', 'GENDER', 'Car_Owner', 'Propert_Owner', 'CHILDREN', 'Annual_income', 'Type_Income', 'EDUCATION', 'Marital_status', 'Housing_type', 'Birthday_count', 'Employed_days', 'Mobile_phone', 'Work_Phone', 'Phone', 'EMAIL_ID', 'Type_Occupation', 'Family_Members']

Columnas de df_label:
['Ind_ID', 'label']


In [25]:
# Comprobar valores nulos
print("Nulos en df:", df["Ind_ID"].isna().sum())
print("Nulos en df_label:", df_label["Ind_ID"].isna().sum())

# Comprobar identificadores duplicados
print("Duplicados en df:", df["Ind_ID"].duplicated().sum())
print("Duplicados en df_label:", df_label["Ind_ID"].duplicated().sum())

# Comprobar cantos identificadores coinciden
print("IDs comúns:", len(set(df["Ind_ID"]) & set(df_label["Ind_ID"])))

Nulos en df: 0
Nulos en df_label: 0
Duplicados en df: 0
Duplicados en df_label: 0
IDs comúns: 1548


Haremos un merge tipo INNER porque sólo nos interesan los solicitantes que tienen su ID en el dataset con la variable objeivo. No nos interesan personas que no aparezcan como aceptadas o rechazadas en esta tabla.

In [5]:
df_final = pd.merge(
    df,
    df_label,
    on="Ind_ID",
    how="inner",
    validate="one_to_one"
)

In [27]:
print("Dimensións de df:", df.shape)
print("Dimensións de df_label:", df_label.shape)
print("Dimensións de df_final:", df_final.shape)

print("Filas completamente duplicadas:", df_final.duplicated().sum())

display(df_final.head())

Dimensións de df: (1548, 18)
Dimensións de df_label: (1548, 2)
Dimensións de df_final: (1548, 19)
Filas completamente duplicadas: 0


,Ind_ID,GENDER,Car_Owner,Propert_Owner,CHILDREN,Annual_income,Type_Income,EDUCATION,Marital_status,Housing_type,Birthday_count,Employed_days,Mobile_phone,Work_Phone,Phone,EMAIL_ID,Type_Occupation,Family_Members,label
0,5008827,M,Y,Y,0,180000.0,Pensioner,Higher education,Married,House / apartment,-18772.0,365243,1,0,0,0,NaN,2,1
1,5009744,F,Y,N,0,315000.0,Commercial associate,Higher education,Married,House / apartment,-13557.0,-586,1,1,1,0,NaN,2,1
2,5009746,F,Y,N,0,315000.0,Commercial associate,Higher education,Married,House / apartment,NaN,-586,1,1,1,0,NaN,2,1
3,5009749,F,Y,N,0,NaN,Commercial associate,Higher education,Married,House / apartment,-13557.0,-586,1,1,1,0,NaN,2,1
4,5009752,F,Y,N,0,315000.0,Commercial associate,Higher education,Married,House / apartment,-13557.0,-586,1,1,1,0,NaN,2,1


### Correccion de nulos en los otros campos

Antes de realizar el preprocesamiento de datos, se analizaron el número de nulos apra cada variable y se decidieron tomar disintos tratamientos, como se explica a continuación:

- GENDER: Hay 7 nulos, y se opta por eliminar esas filas. Su eliminación no supone una pérdida significativa de la información. 

- ANNUAL INCOME: Hay 23 nulos, y se eliminan por la misma razón.

- BIRTHDAY COUNT: hay 22 nulos, y se eliminan por la misma razón.

- TYPE OCCUPATION: Se conservan los 488 registros porque suponen un porcentaje cercano al tercio. No se pueden eliminar. En los siguientes apartados se deliberará qué hacer con estos datos. se puede crear la catagoría "Desconocido" y meter en ella a los nulos de este campo.

In [ ]:
df_final.info()

In [7]:
# 1. Eliminar filas con GENDER nulo
df_final = df_final.dropna(subset=["GENDER"]).copy()
# 1. Eliminar filas con ANNUAL INCOME nulo
df_final = df_final.dropna(subset=["Annual_income"]).copy()
# 1. Eliminar filas con BIRTHDAY COUNT nulo
df_final = df_final.dropna(subset=["Birthday_count"]).copy()

In [8]:
# Valores nulos restantes
print(df_final.isnull().sum())

# Dimensións do DataFrame
print("Dimensións finais:", df_final.shape)

Ind_ID               0
GENDER               0
Car_Owner            0
Propert_Owner        0
CHILDREN             0
Annual_income        0
Type_Income          0
EDUCATION            0
Marital_status       0
Housing_type         0
Birthday_count       0
Employed_days        0
Mobile_phone         0
Work_Phone           0
Phone                0
EMAIL_ID             0
Type_Occupation    471
Family_Members       0
label                0
dtype: int64
Dimensións finais: (1496, 19)


### Eliminacion y/o transformacion de campos

Se revisaron variables del dataset para ver si se podían mejorar algunas y eliminar las que no aportaban nada.

Se decide excluír a las siguientes variables:

- Ind_ID: variable con todos los valores únicos, no aporta información
- Mobile_phone: variable que sólo tiene unos, no aporta información
- Birthday_count: se crea la variable `Age` a partir de ella y se elimina.
- Employed_days: Se cean las variables

In [32]:
df_final["Age"] = -df_final["Birthday_count"] / 365.25

In [36]:
# Comprobar cantos rexistros teñen o código especial
print((df_final["Employed_days"] == 365243).sum())

260


In [37]:
# Identificar situacións co código especial
df_final["Employment_unknown"] = (
    df_final["Employed_days"] == 365243
).astype(int)

# Calcular anos traballados, excluíndo o código especial
df_final["Years_employed"] = (
    -df_final["Employed_days"].where(
        df_final["Employed_days"] != 365243
    ) / 365.25
)

In [39]:
df_final.head()

,Ind_ID,GENDER,Car_Owner,Propert_Owner,CHILDREN,Annual_income,Type_Income,EDUCATION,Marital_status,Housing_type,...,Mobile_phone,Work_Phone,Phone,EMAIL_ID,Type_Occupation,Family_Members,label,Age,Years_employed,Employment_unknown
0,5008827,M,Y,Y,0,180000.0,Pensioner,Higher education,Married,House / apartment,...,1,0,0,0,NaN,2,1,51.394935,NaN,1
1,5009744,F,Y,N,0,315000.0,Commercial associate,Higher education,Married,House / apartment,...,1,1,1,0,NaN,2,1,37.117043,1.604381,0
2,5009746,F,Y,N,0,315000.0,Commercial associate,Higher education,Married,House / apartment,...,1,1,1,0,NaN,2,1,NaN,1.604381,0
3,5009749,F,Y,N,0,NaN,Commercial associate,Higher education,Married,House / apartment,...,1,1,1,0,NaN,2,1,37.117043,1.604381,0
4,5009752,F,Y,N,0,315000.0,Commercial associate,Higher education,Married,House / apartment,...,1,1,1,0,NaN,2,1,37.117043,1.604381,0


In [40]:
print(df_final[[
    "Employed_days",
    "Years_employed",
    "Employment_unknown"
]].head(10))

print(df_final["Employment_unknown"].value_counts())

    Employed_days  Years_employed  Employment_unknown
0          365243             NaN                   1
1            -586        1.604381                   0
2            -586        1.604381                   0
3            -586        1.604381                   0
4            -586        1.604381                   0
6            -586        1.604381                   0
7          365243             NaN                   1
8            -678        1.856263                   0
9            -678        1.856263                   0
10           -678        1.856263                   0
Employment_unknown
0    1281
1     260
Name: count, dtype: int64


In [41]:
display(df_final.head())
df_final.info()

,Ind_ID,GENDER,Car_Owner,Propert_Owner,CHILDREN,Annual_income,Type_Income,EDUCATION,Marital_status,Housing_type,...,Mobile_phone,Work_Phone,Phone,EMAIL_ID,Type_Occupation,Family_Members,label,Age,Years_employed,Employment_unknown
0,5008827,M,Y,Y,0,180000.0,Pensioner,Higher education,Married,House / apartment,...,1,0,0,0,NaN,2,1,51.394935,NaN,1
1,5009744,F,Y,N,0,315000.0,Commercial associate,Higher education,Married,House / apartment,...,1,1,1,0,NaN,2,1,37.117043,1.604381,0
2,5009746,F,Y,N,0,315000.0,Commercial associate,Higher education,Married,House / apartment,...,1,1,1,0,NaN,2,1,NaN,1.604381,0
3,5009749,F,Y,N,0,NaN,Commercial associate,Higher education,Married,House / apartment,...,1,1,1,0,NaN,2,1,37.117043,1.604381,0
4,5009752,F,Y,N,0,315000.0,Commercial associate,Higher education,Married,House / apartment,...,1,1,1,0,NaN,2,1,37.117043,1.604381,0


<class 'pandas.DataFrame'>
Index: 1541 entries, 0 to 1547
Data columns (total 22 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Ind_ID              1541 non-null   int64  
 1   GENDER              1541 non-null   str    
 2   Car_Owner           1541 non-null   str    
 3   Propert_Owner       1541 non-null   str    
 4   CHILDREN            1541 non-null   int64  
 5   Annual_income       1518 non-null   float64
 6   Type_Income         1541 non-null   str    
 7   EDUCATION           1541 non-null   str    
 8   Marital_status      1541 non-null   str    
 9   Housing_type        1541 non-null   str    
 10  Birthday_count      1519 non-null   float64
 11  Employed_days       1541 non-null   int64  
 12  Mobile_phone        1541 non-null   int64  
 13  Work_Phone          1541 non-null   int64  
 14  Phone               1541 non-null   int64  
 15  EMAIL_ID            1541 non-null   int64  
 16  Type_Occupation     10

In [42]:
# Valores únicos das variables binarias
for col in ["Mobile_phone", "Work_Phone", "Phone", "EMAIL_ID",
            "GENDER", "Car_Owner", "Propert_Owner"]:
    print(f"\n{col}:")
    print(df_final[col].value_counts(dropna=False))

# Número de valores distintos de cada columna
display(df_final.nunique().sort_values())


Mobile_phone:
Mobile_phone
1    1541
Name: count, dtype: int64

Work_Phone:
Work_Phone
0    1220
1     321
Name: count, dtype: int64

Phone:
Phone
0    1066
1     475
Name: count, dtype: int64

EMAIL_ID:
EMAIL_ID
0    1399
1     142
Name: count, dtype: int64

GENDER:
GENDER
F    973
M    568
Name: count, dtype: int64

Car_Owner:
Car_Owner
N    922
Y    619
Name: count, dtype: int64

Propert_Owner:
Propert_Owner
Y    1007
N     534
Name: count, dtype: int64


Mobile_phone             1
GENDER                   2
Propert_Owner            2
Car_Owner                2
Phone                    2
Work_Phone               2
EMAIL_ID                 2
label                    2
Employment_unknown       2
Type_Income              4
Marital_status           5
EDUCATION                5
CHILDREN                 6
Housing_type             6
Family_Members           7
Type_Occupation         18
Annual_income          115
Years_employed         952
Employed_days          953
Birthday_count        1266
Age                   1266
Ind_ID                1541
dtype: int64

In [43]:
df_final["Mobile_phone"].value_counts()

Mobile_phone
1    1541
Name: count, dtype: int64

In [44]:
df_final_col_eliminadas = df_final.drop(
    columns=[
        "Ind_ID",
        "Mobile_phone",
        "Birthday_count",
        "Employed_days"
    ]
).copy()

In [45]:
print("DataFrame orixinal:", df_final.shape)
print("DataFrame para modelado:", df_final_col_eliminadas.shape)

display(df_final_col_eliminadas.head())

DataFrame orixinal: (1541, 22)
DataFrame para modelado: (1541, 18)


,GENDER,Car_Owner,Propert_Owner,CHILDREN,Annual_income,Type_Income,EDUCATION,Marital_status,Housing_type,Work_Phone,Phone,EMAIL_ID,Type_Occupation,Family_Members,label,Age,Years_employed,Employment_unknown
0,M,Y,Y,0,180000.0,Pensioner,Higher education,Married,House / apartment,0,0,0,NaN,2,1,51.394935,NaN,1
1,F,Y,N,0,315000.0,Commercial associate,Higher education,Married,House / apartment,1,1,0,NaN,2,1,37.117043,1.604381,0
2,F,Y,N,0,315000.0,Commercial associate,Higher education,Married,House / apartment,1,1,0,NaN,2,1,NaN,1.604381,0
3,F,Y,N,0,NaN,Commercial associate,Higher education,Married,House / apartment,1,1,0,NaN,2,1,37.117043,1.604381,0
4,F,Y,N,0,315000.0,Commercial associate,Higher education,Married,House / apartment,1,1,0,NaN,2,1,37.117043,1.604381,0
